# LLM as a Judge (GPT-4)

### Setup & imports

In [2]:
# Setup & imports
import pandas as pd

### Setup & imports

In [ ]:
# Setup & imports

import os
import base64
from openai import AzureOpenAI

endpoint = os.getenv("ENDPOINT_URL", "")
deployment = os.getenv("DEPLOYMENT_NAME", "gpt-4.1")
subscription_key = os.getenv("AZURE_OPENAI_API_KEY", "")  


# Initialize Azure OpenAI client with key-based authentication",
client = AzureOpenAI(
    azure_endpoint=endpoint,
    api_key=subscription_key,
    api_version="2025-01-01-preview",
)

# IMAGE_PATH = "YOUR_IMAGE_PATH"\n"
# encoded_image = base64.b64encode(open(IMAGE_PATH, 'rb').read()).decode('ascii')\n"

### System Prompt

In [3]:

#prompt builder

def build_prompt(pred, act):

    system_prompt = '''You are to act as a judge, evaluating whether a predicted answer aligns closely enough with a provided answer to be considered correct.

To assess if the predicted answer is correct:  
- Variations in language are acceptable as long as the underlying information and meaning are similar.  
- Phrasing or stylistic differences should not affect the judgment.  
- Additional context and details in the predicted answer are acceptable.
- Focus on the core facts, intent, and substance of the answer to determine similarity.

# Steps
1. Compare the **provided answer** and the **predicted answer**.  
2. Identify if the key information, intent, and meaning are equivalent.  
3. Ignore minor wording or phrasing differences that do not alter the meaning.  
4. Give a final assessment: "Yes" if the predicted answer is sufficiently similar, and "No" otherwise.

# Output Format
Your response should use the following format **exactly**:
```
Evaluation: [Yes/No]
Reasoning: [Explain the reasoning behind your evaluation by briefly highlighting similarities or differences between the two answers.]
```

# Example
Input:
**Provided Answer**: "The capital of France is Paris."  
**Predicted Answer**: "Paris is the capital city of France."

Output:
```
Evaluation: Yes
Reasoning: The predicted answer conveys the same information as the provided answer, with no significant difference in meaning despite slight variation in phrasing.
```

Input:
**Provided Answer**: "Water boils at 100 degrees Celsius under standard atmospheric pressure."  
**Predicted Answer**: "Water freezes at 100 degrees Celsius at normal pressure."

Output:
```
Evaluation: No
Reasoning: The predicted answer is factually incorrect as it states water freezes at 100 degrees Celsius, which deviates from the provided answer about boiling.
```'''

    chat_prompt = [
        {
            "role": "system",
            "content": [
                {
                    "type": "text",
                    "text": system_prompt
                }
            ]
        },
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": f'''**Provided Answer**: "{act}"
**Predicted Answer**: "{pred}"'''
                }
            ]
        }
    ]
    return chat_prompt


In [4]:
# Modeling & evaluation
def openai_api(pred, act):
    # Include speech result if speech is enabled
    messages = build_prompt(pred, act)
    
    # Generate the completion
    completion = client.chat.completions.create(
        model=deployment,
        messages=messages,
        max_tokens=2000,
        temperature=0.7,
        top_p=0.95,
        frequency_penalty=0,
        presence_penalty=0,
        stop=None,
        stream=False
    )
    
    return completion.choices[0].message.content

### Data loading

In [ ]:
# Data loading
answers = pd.read_json('./runs/Qwen2.5-VL-7B-Instruct-MLLM-only-answers-custom-retrieval-openai-with-system-prompt.json')

In [10]:
# Computation
results = []
for index, row in answers.iterrows():
    result = openai_api(pred=row['Answer'], act=row['Gold Answer'])

    parts = result.split("Reasoning:")  
    evaluation_part = parts[0].replace("Evaluation:", "").strip()[5:] 
    reasoning_part = parts[1].strip()

    results.append([evaluation_part, reasoning_part])

In [11]:
# Calculate accuracy  
total_evaluations = len(results)  
yes_count = sum(1 for item in results if item[0].strip().lower() == "yes")  
  
accuracy = (yes_count / total_evaluations) * 100  # Convert to percentage  
  
# Print the accuracy  
print(f"Accuracy: {accuracy:.2f}%")  

Accuracy: 48.33%


In [12]:
# Computation
df = pd.DataFrame(results, columns=['evaluation', 'reasoning'])

In [ ]:
# Computation
df.to_json('./runs/Qwen2.5-VL-7B-Instruct-MLLM-only-answers-custom-retrieval-openai-with-system-prompt.json', orient='records', indent=4)